## In this notebook I'll solve linear equation systems

I'll go into details on
- Linear Systems
- Pivot variables and pivot columns
- Free variables and free columns
- Reduced row echelon Form (rref)
- Column space
- Nullspace
- Nullspace basis vectors (s1, s2, ... special solutions)
- Particular solutions (xp)
- Complete solutions (x = xp + c1s1 + c2s2 ...)

The program should also reveal how many solutions a given system has (0, 1, or infinite many solutions)

I'll also augment my data with markdowns, to make it easier to follow, and for future reference

I'm using the classic Gilbert Stang approach to find the complete solutions

In [1]:
import os 
import numpy as np

In [117]:
A = np.array([[1,2,3,5],[2,4,8,12],[3,6,7,13]], dtype=float)
b = np.array([4, 10, 10], dtype=float).reshape(-1, 1)
m = len(A)
n = len(A[0])
print("The Coefficient matrix")
print(A)
print("Right hand side vector")
print(b)

The Coefficient matrix
[[ 1.  2.  3.  5.]
 [ 2.  4.  8. 12.]
 [ 3.  6.  7. 13.]]
Right hand side vector
[[ 4.]
 [10.]
 [10.]]


Step 1: Augment the matrix, and establish its reduced row echelon form (RREF) using elementary row operations 

To solve Ax = b, I need to combine the coefficient matrix A with b right hand side vector into an augmented matrix [A | b]

RREF:
Elementary row operations so the pivot columns contain the leading 1s of each non-zero row. The correspond to pivot variable
Columns without pivots correspond to free variables. This gives us our degree of freedom for infinitely many solutions.

In [118]:
augmented_matrix = np.concatenate([A, b], axis=1)
print(augmented_matrix)

[[ 1.  2.  3.  5.  4.]
 [ 2.  4.  8. 12. 10.]
 [ 3.  6.  7. 13. 10.]]


Reduced row echelon form calculation.

In [121]:
def reduced_row_echelon_form(A: np.array):
    R = A.copy()
    row = 0
    for col in range(0, n):
        if not (np.any(R[row:, col] != 0)):
            continue
        R[row:, :] = R[row + np.argsort(R[row:, col])[::-1], :]

        for i in range(0, row):
                R[i] = R[i] - (R[i, col] / R[row, col]) * R[row]
        for i in range(row+1, m):
            R[i] = R[i] - (R[i, col] / R[row, col]) * R[row]
        R[row] = (R[row] / (R[row, col]))
        R[R==0] = 0.0

        row +=1
    return R

R = reduced_row_echelon_form(augmented_matrix)
print("Reduced row echelon form of the augmented matrix")
print(R)

Reduced row echelon form of the augmented matrix
[[1. 2. 0. 2. 1.]
 [0. 0. 1. 1. 1.]
 [0. 0. 0. 0. 0.]]


Step 2: Find the pivot and free column indices, identify the nullspace basis vectors

To understand our system, we need to distinguish between pivot and free variables.
- Pivot columns correspond to the leading 1s in the Reduced Row Echelon Form 
- Free columns gives us the degree of freedom when there are infinitely many solutions, these are the nullspace basis vectors

The following two functions:
- **get_pivot_and_free_indices** Identifies the pivot and free column indices based on leading 1s from RREF of the augmented matrix
- **nullspace_basis_vectors** These vectors are the basis vectors for the nullspace, they span the space. They are the special solutions (s_1, s_2, etc)

In [124]:
def get_pivot_and_free_indices(R: np.array):
    m, n = R.shape
    pivot_indices = []
    row = 0

    for col in range(n):
        if row < m and R[row, col] == 1:
            pivot_indices.append(col)
            row += 1

    pivot_indices_set = set(pivot_indices)
    free_indices = [i for i in range(n) if i not in pivot_indices_set]

    return pivot_indices, free_indices

# The last column is taken out since it's the right hand side vector
get_pivot_and_free_indices(R[:, :-1])

([0, 2], [1, 3])

In [126]:
def nullspace_basis_vectors(R: np.array):
    _, n = R.shape
    pivot_indices, free_indices = get_pivot_and_free_indices(R)
    s = []
    for free_index in free_indices:
        nullspace_vector = np.zeros(n, dtype=float)
        nullspace_vector[free_index] = 1

        j = 0
        while j < len(pivot_indices):
            pivot_index = pivot_indices[j]
            nullspace_vector[pivot_index] = -R[j, free_index]
            j += 1
        
        nullspace_vector[nullspace_vector == 0] = 0.0
        s.append(nullspace_vector)
    return s

s = nullspace_basis_vectors(R[:, :-1])
print(s)

[array([-2.,  1.,  0.,  0.]), array([-2.,  0., -1.,  1.])]


## Step 3: Determining the Number of Solutions and the Complete Solution

With our RREF matrix, pivot/free indices, and nullspace basis vectors ready, we can write a final function to bring everything together.

The **findSolutions** function handles all three possible outcomes for a linear system:
-  Zero Solutions: If any row in R consists entirely of zeros, but the corresponding entry in the augmented column c is non-zero (0 = non-zero), the system is inconsistent and yields 0 solutions.
-  1 Solution: If the system is consistent and has no free variables, the pivot variables completely determine the unique solution (x = x_p).
-  Infinitely Many Solutions: If the system is consistent and contains free variables, we get an infinite family of solutions. We express the Complete Solution as the sum of the particular solution and a linear combination of the nullspace basis vectors (special solutions):
   x = x_p + c_1 s_1 + c_2 s_2 + ...

In [128]:
def findSolutions(R_augmented: np.array):
    R = R_augmented[:, :-1]
    c = R_augmented[:, -1]


    #find which rows are zeros vectors
    zero_row_mask = np.all(R==0, axis=1)
    if np.any(c[zero_row_mask]!=0):
        return 0


    non_zero_rows = ~zero_row_mask
    R = R[non_zero_rows, :].copy()
    c = c[non_zero_rows].copy()
    pivot_ind, free_ind = get_pivot_and_free_indices(R)
    xp = np.zeros(shape=n)
    xp[pivot_ind] = c
    if len(free_ind) == 0:
        print(f"The linear system has only 1 solution: {xp}")
    else:
        null_bases_vectors = nullspace_basis_vectors(R)
        print("The linear system has infinite number of solutions")
        print(f"X complete = X particular (1) + c * X nullspace, where c is any number")
        print(f"X particular vector {xp}, X nullspace basis vectors {null_bases_vectors}")


findSolutions(R)

The linear system has infinite number of solutions
X complete = X particular (1) + c * X nullspace, where c is any number
X particular vector [1. 0. 1. 0.], X nullspace basis vectors [array([-2.,  1.,  0.,  0.]), array([-2.,  0., -1.,  1.])]


In [ ]:
# A = np.array([[1,2,3],[2,5,8],[3,1,8]], dtype=float)
# A = np.array([[0,1,2],[1,2,3],[2,4,7]], dtype=float)
# A = np.array([[1,7,3,35],[2,14,6,70],[2,14,9,97]], dtype=float)